In [7]:
import geopandas as gpd
import pandas as pd

# 데이터

In [8]:
data=pd.read_csv("태블로용_주소_귀농유형_클러스터.csv", encoding="utf-8-sig")
data.head(2)

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리,법정동코드
0,경상북도 경주시 노동동,영농상속형,교육,10,경상북도,경주시,노동동,NaN,47130106
1,경상북도 경주시 노동동,영농상속형,교통,6,경상북도,경주시,노동동,NaN,47130106


## shp파일

In [9]:
dong=gpd.read_file("emd_202307_utf8_wgs84/emd_202307_utf8_wgs84.shp", encoding="utf-8-sig")
dong.head()

,EMD_CD,EMD_ENG_NM,EMD_KOR_NM,geometry
0,11110101,Cheongun-dong,청운동,"POLYGON ((126.97556 37.58968, 126.97549 37.589..."
1,11110102,Singyo-dong,신교동,"POLYGON ((126.97031 37.58418, 126.97033 37.584..."
2,11110103,Gungjeong-dong,궁정동,"POLYGON ((126.974 37.58654, 126.97401 37.58653..."
3,11110104,Hyoja-dong,효자동,"POLYGON ((126.97356 37.58323, 126.97355 37.582..."
4,11110105,Changseong-dong,창성동,"POLYGON ((126.97353 37.58182, 126.97354 37.581..."


In [10]:
li=gpd.read_file("리 (2)/li.shp", encoding="cp949")
li.head()

,LI_CD,LI_ENG_NM,LI_KOR_NM,geometry
0,2671025021,Dongbu-ri,동부리,"POLYGON ((1156572.843 1696804.747, 1156605.133..."
1,2671025022,Gyo-ri,교리,"POLYGON ((1155731.257 1697372.017, 1155813.409..."
2,2671025023,Sincheon-ri,신천리,"POLYGON ((1157572.293 1695843.731, 1157584.155..."
3,2671025024,Jukseong-ri,죽성리,"POLYGON ((1158010.875 1696998.51, 1158017.903 ..."
4,2671025025,Seobu-ri,서부리,"POLYGON ((1155615.822 1696551.022, 1155624.394..."


# shp파일

In [11]:
# LI_CD 길이별 개수 확인
length_8 = len(li[li['LI_CD'].str.len() == 8])
length_10 = len(li[li['LI_CD'].str.len() == 10])

print(f"8자리 LI_CD 개수: {length_8}")
print(f"10자리 LI_CD 개수: {length_10}")


8자리 LI_CD 개수: 0
10자리 LI_CD 개수: 15161


In [12]:
# LI_CD 길이별 개수 확인
length_8 = len(dong[dong['EMD_CD'].str.len() == 8])
length_10 = len(dong[dong['EMD_CD'].str.len() == 10])

print(f"8자리 EMD_CD 개수: {length_8}")
print(f"10자리 EMD_CD 개수: {length_10}")


8자리 EMD_CD 개수: 5065
10자리 EMD_CD 개수: 0


# shp파일 전처리

두 파일에 행정동코드가 10자리, 8자리만 각각있다.  
따라서 태블로용 시각화데이터에 존재하는 행정동코드에 맞는 shp데이터만 취합 후 새로운 shp파일을 생성한다.

In [13]:
li_filtered = li[li['LI_CD'].astype(str).isin(data["법정동코드"].astype(str))]
li_filtered.shape

(3, 4)

In [14]:
dong_filtered = dong[dong['EMD_CD'].astype(str).isin(data["법정동코드"].astype(str))]
dong_filtered.shape

(27, 4)

# 합치기

In [15]:
li_filtered=li_filtered.rename(columns={"LI_CD":"CD", "LI_ENG_NM":"ENG_NM", "LI_KOR_NM":"KOR_NM"})
dong_filtered=dong_filtered.rename(columns={"EMD_CD":"CD", "EMD_ENG_NM":"ENG_NM", "EMD_KOR_NM":"KOR_NM"})

In [16]:
# 좌표계를 WGS 84로 통일
li_filtered = li_filtered.to_crs(li.crs)  # 원본 li 파일의 좌표계로 변환
dong_filtered = dong_filtered.to_crs(li.crs)  # WGS 84 좌표계로 변환

# 이제 두 GeoDataFrame을 연결
combined_gdf = pd.concat([li_filtered, dong_filtered], axis=0)

In [19]:
# shp 파일로 저장
combined_gdf.to_file("새지도파일/combined_areas.shp", driver='ESRI Shapefile')

In [18]:
# LI_CD 길이별 개수 확인
length_8 = len(combined_gdf[combined_gdf['CD'].str.len() == 8])
length_10 = len(combined_gdf[combined_gdf['CD'].str.len() == 10])

print(f"8자리 LI_CD 개수: {length_8}")
print(f"10자리 LI_CD 개수: {length_10}")


8자리 LI_CD 개수: 27
10자리 LI_CD 개수: 3
